**02_eda.ipynb**
+ bureau.csv analizi
+ previous_application.csv analizi
+ join kontrolleri
+ feature store kontrolleri
+ modelleme öncesi veri doğrulama

In [9]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print(project_root)

c:\Users\ASUS\creditguard-ai


In [3]:
import pandas as pd

bureau = pd.read_csv(
    "../data/raw/bureau.csv"
)

print(bureau.shape)
bureau.head()

(1716428, 17)


,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,-21,NaN


In [4]:
bureau.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1716428 entries, 0 to 1716427
Data columns (total 17 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   SK_ID_CURR              int64  
 1   SK_ID_BUREAU            int64  
 2   CREDIT_ACTIVE           object 
 3   CREDIT_CURRENCY         object 
 4   DAYS_CREDIT             int64  
 5   CREDIT_DAY_OVERDUE      int64  
 6   DAYS_CREDIT_ENDDATE     float64
 7   DAYS_ENDDATE_FACT       float64
 8   AMT_CREDIT_MAX_OVERDUE  float64
 9   CNT_CREDIT_PROLONG      int64  
 10  AMT_CREDIT_SUM          float64
 11  AMT_CREDIT_SUM_DEBT     float64
 12  AMT_CREDIT_SUM_LIMIT    float64
 13  AMT_CREDIT_SUM_OVERDUE  float64
 14  CREDIT_TYPE             object 
 15  DAYS_CREDIT_UPDATE      int64  
 16  AMT_ANNUITY             float64
dtypes: float64(8), int64(6), object(3)
memory usage: 222.6+ MB


In [5]:
bureau[
    [
        "SK_ID_CURR",
        "SK_ID_BUREAU",
        "CREDIT_ACTIVE",
        "AMT_CREDIT_SUM",
        "AMT_CREDIT_SUM_DEBT"
    ]
].head()

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT
0,215354,5714462,Closed,91323.0,0.0
1,215354,5714463,Active,225000.0,171342.0
2,215354,5714464,Active,464323.5,NaN
3,215354,5714465,Active,90000.0,NaN
4,215354,5714466,Active,2700000.0,NaN


In [10]:


bureau = pd.read_csv(
    "../data/raw/bureau.csv"
)

from src.features.build_bureau_features import (
    create_bureau_features
)

bureau_features = create_bureau_features(
    bureau
)

bureau_features.head()

,SK_ID_CURR,bureau_total_credit,bureau_total_debt,bureau_active_loans,bureau_closed_loans,bureau_overdue_amount,bureau_debt_credit_ratio
0,100001,1453365.000,596686.5,3,4,0.0,0.410555
1,100002,865055.565,245781.0,2,6,0.0,0.284122
2,100003,1017400.500,0.0,1,3,0.0,0.000000
3,100004,189037.800,0.0,0,2,0.0,0.000000
4,100005,657126.000,568408.5,2,1,0.0,0.864992


In [11]:
bureau_features.shape

(305811, 7)

In [12]:
train = pd.read_csv(
    "../data/raw/application_train.csv"
)

print(train.shape)

(307511, 122)


In [15]:
train_bureau = train.merge(
    bureau_features,
    on="SK_ID_CURR",
    how="left"
)

print(train_bureau.shape)
# 122 orijinal kolon + 6 yeni bureau feature = 128 kolon

(307511, 128)


In [16]:
# Muhtemelen bazı müşterilerin bureau kaydı olmadığı için eksikler göreceğiz.
train_bureau[
    [
        "bureau_total_credit",
        "bureau_total_debt",
        "bureau_active_loans",
        "bureau_closed_loans"
    ]
].isnull().mean()

bureau_total_credit    0.143149
bureau_total_debt      0.143149
bureau_active_loans    0.143149
bureau_closed_loans    0.143149
dtype: float64

**yaklaşık:** %14.3 müşterinin bureau geçmişi yok.

Anlamı: Kredi bürosunda kayıtlı geçmiş kredi bilgisi bulunmuyor.


İleride eklenecek feature: **has_bureau_history**

In [17]:
train_bureau[
    [
        "bureau_total_credit",
        "bureau_total_debt",
        "bureau_debt_credit_ratio"
    ]
].describe()

c:\Users\ASUS\creditguard-ai\venv\Lib\site-packages\numpy\core\_methods.py:49: RuntimeWarning: invalid value encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)


,bureau_total_credit,bureau_total_debt,bureau_debt_credit_ratio
count,2.634910e+05,2.634910e+05,2.624720e+05
mean,1.955807e+06,6.406503e+05,NaN
std,4.101728e+06,1.633961e+06,NaN
min,0.000000e+00,-6.981558e+06,-inf
25%,3.433773e+05,0.000000e+00,0.000000e+00
50%,9.617040e+05,1.690200e+05,2.091433e-01
75%,2.297721e+06,6.600621e+05,4.830696e-01
max,1.017958e+09,3.344983e+08,inf


In [ ]:
previous = pd.read_csv(
    "../data/raw/previous_application.csv" 
      #"Bu müşteri geçmişte Home Credit'e kaç kez başvurdu?"
)

print(previous.shape)

(1670214, 37)


In [19]:
previous.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1670214 entries, 0 to 1670213
Data columns (total 37 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   SK_ID_PREV                   1670214 non-null  int64  
 1   SK_ID_CURR                   1670214 non-null  int64  
 2   NAME_CONTRACT_TYPE           1670214 non-null  object 
 3   AMT_ANNUITY                  1297979 non-null  float64
 4   AMT_APPLICATION              1670214 non-null  float64
 5   AMT_CREDIT                   1670213 non-null  float64
 6   AMT_DOWN_PAYMENT             774370 non-null   float64
 7   AMT_GOODS_PRICE              1284699 non-null  float64
 8   WEEKDAY_APPR_PROCESS_START   1670214 non-null  object 
 9   HOUR_APPR_PROCESS_START      1670214 non-null  int64  
 10  FLAG_LAST_APPL_PER_CONTRACT  1670214 non-null  object 
 11  NFLAG_LAST_APPL_IN_DAY       1670214 non-null  int64  
 12  RATE_DOWN_PAYMENT            774370 non-nu

In [20]:
from src.features.build_previous_features import (
    create_previous_features
)

previous_features = create_previous_features(
    previous
)

previous_features.shape

(338857, 9)

**application_train :** 307.511 müşteri

**bureau            :** 305.811 müşteri

**previous_app      :** 338.857 müşteri

In [21]:
previous_features.head()

,SK_ID_CURR,prev_application_count,prev_approved_count,prev_refused_count,prev_canceled_count,prev_avg_application_amount,prev_avg_credit_amount,approval_rate,refusal_rate
0,100001,1,1,0,0,24835.50,23787.00,1.0,0.0
1,100002,1,1,0,0,179055.00,179055.00,1.0,0.0
2,100003,3,3,0,0,435436.50,484191.00,1.0,0.0
3,100004,1,1,0,0,24282.00,20106.00,1.0,0.0
4,100005,2,1,0,1,22308.75,20076.75,0.5,0.0


In [22]:
previous_features[
    [
        "prev_application_count",
        "approval_rate",
        "refusal_rate"
    ]
].describe()

,prev_application_count,approval_rate,refusal_rate
count,338857.000000,338857.000000,338857.00000
mean,4.928964,0.744495,0.11142
std,4.220716,0.263250,0.18373
min,1.000000,0.000000,0.00000
25%,2.000000,0.500000,0.00000
50%,4.000000,0.777778,0.00000
75%,7.000000,1.000000,0.20000
max,77.000000,1.000000,1.00000


### 📊 Başvuru Sayısı İstatistikleri (`prev_application_count`)

* **Ortalama (Mean):** 4.93
* **Medyan (Median):** 4
* **Maksimum (Max):** 77

> **💡 Özet Bulgular:**
> * **Ortalama Müşteri:** Yaklaşık **5 kez** başvurmuş.
> * **Uç Değerler:** Bazı müşterilerde bu sayı **77 başvuruya** kadar çıkmaktadır.

---

### 📈 Başvuru Durum Oranları

* **Onaylanma Oranı (Approval Rate):** `Mean = 0.744` (Başvuruların **%74.4'ü** onaylanmış)
* **Reddedilme Oranı (Refusal Rate):** `Mean = 0.111` (Ortalama reddedilme oranı **%11.1**)


In [1]:
import pandas as pd

df = pd.read_parquet(
    "../data/processed/train_feature_store.parquet"
)

print(df.shape)

(307511, 150)


In [2]:
df.columns[-20:].tolist()

['credit_per_child',
 'income_credit_difference',
 'annuity_credit_ratio',
 'is_car_owner',
 'is_realty_owner',
 'is_employed',
 'bureau_total_credit',
 'bureau_total_debt',
 'bureau_active_loans',
 'bureau_closed_loans',
 'bureau_overdue_amount',
 'bureau_debt_credit_ratio',
 'prev_application_count',
 'prev_approved_count',
 'prev_refused_count',
 'prev_canceled_count',
 'prev_avg_application_amount',
 'prev_avg_credit_amount',
 'approval_rate',
 'refusal_rate']

**Bureau Feature'ları**

+ bureau_total_credit
+ bureau_total_debt
+ bureau_active_loans
+ bureau_closed_loans
+ bureau_overdue_amount
+ bureau_debt_credit_ratio

> Model, "Bu müşterinin başka bankalardaki borç durumu nedir?" sorusunu görebilecek.

**Previous Application Feature'ları**
+ prev_application_count
+ prev_approved_count
+ prev_refused_count
+ approval_rate
+ refusal_rate

> Model , ""Bu müşteri geçmişte kredi başvurularında nasıl davranmış?" bilgisini görebilecek."

In [3]:
df.select_dtypes(include="object").columns.tolist()

['NAME_CONTRACT_TYPE',
 'CODE_GENDER',
 'FLAG_OWN_CAR',
 'FLAG_OWN_REALTY',
 'NAME_TYPE_SUITE',
 'NAME_INCOME_TYPE',
 'NAME_EDUCATION_TYPE',
 'NAME_FAMILY_STATUS',
 'NAME_HOUSING_TYPE',
 'OCCUPATION_TYPE',
 'WEEKDAY_APPR_PROCESS_START',
 'ORGANIZATION_TYPE',
 'FONDKAPREMONT_MODE',
 'HOUSETYPE_MODE',
 'WALLSMATERIAL_MODE',
 'EMERGENCYSTATE_MODE']

In [6]:
import numpy as np
df = pd.read_parquet(
    "../data/processed/train_feature_store.parquet"
)

print(
    np.isinf(
        df.select_dtypes(include=np.number)
    ).sum().sum()
)

0
